# 04. Step 2: Category & Sentiment Classification

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook trains and evaluates **Step 2** of the ACOS framework:
- **Model Architecture (`CategorySentiClassification`):** Uses BERT with candidate aspect and opinion span representations to jointly predict aspect categories and sentiment polarities for each candidate pair.
- **Model Checkpointing:** Saves the best fine-tuned model checkpoint (`pytorch_model.bin`, `config.json`, `vocab.txt`) to `checkpoints/step2_best/` based on validation Micro-F1.
- **Evaluation on Pipeline Candidate Pairs:** Evaluates candidate pairs generated from Step 1 (`[domain]_test_pair_1st.tsv`) and produces complete quadruple predictions (`result.txt`), training curves, and metric CSVs.

## 1. Environment & Module Imports

In [ ]:
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3
import os
import sys
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm, trange

# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from modeling import CategorySentiClassification
from bert_utils.tokenization import BertTokenizer
from bert_utils.optimization import BertAdam
from run_classifier_dataset_utils import processors, output_modes, convert_examples_to_features_categorysenti
from dataset_utils import read_pair_gold
from eval_metrics import pair_eval

# 3. Import colab_utils with fallback download
try:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Active PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")


## 2. Configuration & Hyperparameters

In [ ]:
DOMAIN = "rest16"              # 'rest16' or 'laptop'
TASK_NAME = "categorysenti"
MODEL_TYPE = "categorysenti"
DO_TRAIN = True                # Set to False to skip training and evaluate saved checkpoint
DO_EVAL = True
MAX_SEQ_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 5e-5
NUM_TRAIN_EPOCHS = 15          # Default is 30, 15 is great for fast Colab training
WARMUP_PROPORTION = 0.1
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Pretrained BERT Directory
bert_model_dir = os.path.join(base_project_dir, "bert_base_uncased")
download_bert_pretrained(target_dir=bert_model_dir)

data_dir = extract_dir
results_base = os.path.join(base_project_dir, "results")
session_dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)
step2_checkpoint_dir = session_dirs["step2_checkpoint"]

print(f"📁 Step 2 Checkpoint will be saved to: {step2_checkpoint_dir}")

## 3. Data Loading & Feature Processing

In [ ]:
tokenizer = BertTokenizer.from_pretrained(bert_model_dir, do_lower_case=True)
processor = processors[TASK_NAME]()
label_list = processor.get_labels(DOMAIN)
num_labels = len(label_list[0])

print(f"Category-Sentiment Classes ({num_labels}):")
for idx, lbl in enumerate(label_list[0]):
    print(f"  {idx:02d}: {lbl}")

# Check for Step 1 predicted candidate pairs
pipeline_pair_file = os.path.join(data_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
if not os.path.exists(pipeline_pair_file):
    print(f"⚠️ Pipeline pair file {pipeline_pair_file} not found. Fallback to standard test pair.")
    eval_examples = processor.get_test_examples(data_dir, DOMAIN)
else:
    print(f"✅ Using predicted candidate pairs from Step 1: {pipeline_pair_file}")
    eval_examples = processor.get_test_1st_examples(data_dir, DOMAIN)

eval_features = convert_examples_to_features_categorysenti(eval_examples, label_list, MAX_SEQ_LENGTH, tokenizer, output_modes[TASK_NAME], TASK_NAME, domain_type=DOMAIN)

all_input_ids = torch.tensor([f.aspect_input_ids for f in eval_features], dtype=torch.long)
all_input_mask = torch.tensor([f.aspect_input_mask for f in eval_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.aspect_segment_ids for f in eval_features], dtype=torch.long)
all_candidate_aspect = torch.tensor([f.candidate_aspect for f in eval_features], dtype=torch.long)
all_candidate_opinion = torch.tensor([f.candidate_opinion for f in eval_features], dtype=torch.long)
all_label_id = torch.tensor([f.label_id for f in eval_features], dtype=torch.float)
all_tokens_len = torch.tensor([f.tokens_len for f in eval_features], dtype=torch.long)

eval_data = TensorDataset(all_tokens_len, all_input_ids, all_input_mask, all_segment_ids, all_candidate_aspect, all_candidate_opinion, all_label_id)
eval_sampler = SequentialSampler(eval_data)
eval_dataloader = DataLoader(eval_data, sampler=eval_sampler, batch_size=EVAL_BATCH_SIZE)

# Load Ground Truth pairs for evaluation
class ArgsProxy:
    def __init__(self):
        self.bert_model = bert_model_dir
        self.do_lower_case = True
proxy_args = ArgsProxy()

test_pair_gold_file = os.path.join(data_dir, "tokenized_data", f"{DOMAIN}_test_pair.tsv")
with open(test_pair_gold_file, "r", encoding="utf-8") as f:
    eval_gold = read_pair_gold(f.readlines(), proxy_args)

print(f"✅ Evaluated Gold Test Pairs: {len(eval_gold[0])}")

## 4. Model Initialization: `CategorySentiClassification`

In [ ]:
model = CategorySentiClassification.from_pretrained(bert_model_dir, num_labels=num_labels)
model.to(device)
print(f"✅ Initialized CategorySentiClassification model ({sum(p.numel() for p in model.parameters()):,} parameters).")

## 5. Training Loop with Model Checkpointing
Fine-tune the model on Ground Truth training pairs and persist the best checkpoint to `checkpoints/step2_best/`.

In [ ]:
if DO_TRAIN:
    train_examples = processor.get_train_examples(data_dir, DOMAIN)
    train_features = convert_examples_to_features_categorysenti(train_examples, label_list, MAX_SEQ_LENGTH, tokenizer, output_modes[TASK_NAME], TASK_NAME, domain_type=DOMAIN)
    
    tr_input_ids = torch.tensor([f.aspect_input_ids for f in train_features], dtype=torch.long)
    tr_input_mask = torch.tensor([f.aspect_input_mask for f in train_features], dtype=torch.long)
    tr_segment_ids = torch.tensor([f.aspect_segment_ids for f in train_features], dtype=torch.long)
    tr_candidate_aspect = torch.tensor([f.candidate_aspect for f in train_features], dtype=torch.long)
    tr_candidate_opinion = torch.tensor([f.candidate_opinion for f in train_features], dtype=torch.long)
    tr_label_id = torch.tensor([f.label_id for f in train_features], dtype=torch.float)
    tr_tokens_len = torch.tensor([f.tokens_len for f in train_features], dtype=torch.long)
    
    train_data = TensorDataset(tr_tokens_len, tr_input_ids, tr_input_mask, tr_segment_ids, tr_candidate_aspect, tr_candidate_opinion, tr_label_id)
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=TRAIN_BATCH_SIZE)
    
    num_train_optimization_steps = len(train_dataloader) * NUM_TRAIN_EPOCHS
    
    param_optimizer = list(model.named_parameters())
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = BertAdam(optimizer_grouped_parameters, lr=LEARNING_RATE, warmup=WARMUP_PROPORTION, t_total=num_train_optimization_steps)
    
    print(f"🚀 Starting Step 2 Training: {NUM_TRAIN_EPOCHS} Epochs, {len(train_dataloader)} Steps/Epoch...")
    
    class ArgsHelper:
        def __init__(self):
            self.output_dir = session_dirs["logs"]
            self.max_seq_length = MAX_SEQ_LENGTH
    eval_args = ArgsHelper()
    
    import logging
    logger = logging.getLogger("Step2")
    
    best_val_f1 = 0.0
    training_history = []
    
    for epoch in range(1, NUM_TRAIN_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch}/{NUM_TRAIN_EPOCHS}")):
            batch = tuple(t.to(device) for t in batch)
            _len, _ids, _mask, _seg_ids, _cand_a, _cand_o, _lbls = batch
            
            loss, _ = model(
                tokenizer, epoch,
                aspect_input_ids=_ids,
                aspect_token_type_ids=_seg_ids,
                aspect_attention_mask=_mask,
                candidate_aspect=_cand_a,
                candidate_opinion=_cand_o,
                label_id=_lbls
            )
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_dataloader)
        
        # Evaluation
        model.eval()
        val_res = pair_eval(epoch, eval_args, logger, tokenizer, model, eval_dataloader, eval_gold, label_list, device, TASK_NAME, eval_type='test')
        
        val_p = val_res.get('precision', 0.0)
        val_r = val_res.get('recall', 0.0)
        val_f1 = val_res.get('micro-F1', 0.0)
        
        print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f} | Test P: {val_p*100:.2f}% | R: {val_r*100:.2f}% | Micro-F1: {val_f1*100:.2f}%")
        
        training_history.append({
            "epoch": epoch,
            "loss": avg_loss,
            "precision": val_p,
            "recall": val_r,
            "micro-F1": val_f1
        })
        
        # SAVE BEST CHECKPOINT
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            print(f"🔥 New Best Step 2 Test F1: {best_val_f1*100:.2f}%! Saving model checkpoint to {step2_checkpoint_dir}...")
            
            torch.save(model.state_dict(), os.path.join(step2_checkpoint_dir, "pytorch_model.bin"))
            model.config.to_json_file(os.path.join(step2_checkpoint_dir, "config.json"))
            tokenizer.save_vocabulary(step2_checkpoint_dir)
            
            metadata = {
                "epoch": epoch,
                "best_micro_f1": best_val_f1,
                "precision": val_p,
                "recall": val_r,
                "domain": DOMAIN,
                "task": "Step2_Category_Sentiment_Classification"
            }
            with open(os.path.join(step2_checkpoint_dir, "checkpoint_metadata.json"), "w") as mf:
                json.dump(metadata, mf, indent=2)
                
    # Export Training Curves & CSV
    plot_history_path = os.path.join(session_dirs["plots"], "04_step2_training_loss_f1_curve.png")
    csv_history_path = os.path.join(session_dirs["csv"], "step2_training_history.csv")
    plot_training_history(training_history, task_name="Step 2 (Category-Sentiment)", output_plot_path=plot_history_path, output_csv_path=csv_history_path)
    print(f"💾 Saved Step 2 Training History CSV: {csv_history_path}")

## 6. Standalone Checkpoint Loading & Full Quadruple Evaluation
Load the saved checkpoint from `checkpoints/step2_best/` and compute final metrics across all 15 subtasks.

In [ ]:
print(f"📥 Loading best fine-tuned Step 2 checkpoint from: {step2_checkpoint_dir}")
model = CategorySentiClassification.from_pretrained(step2_checkpoint_dir, num_labels=num_labels)
model.to(device)
model.eval()

class ArgsHelper:
    def __init__(self):
        self.output_dir = session_dirs["logs"]
        self.max_seq_length = MAX_SEQ_LENGTH
eval_args = ArgsHelper()

import logging
logger = logging.getLogger("Step2_Final")
final_res = pair_eval('best_checkpoint', eval_args, logger, tokenizer, model, eval_dataloader, eval_gold, label_list, device, TASK_NAME, eval_type='test')

print("\n🏆 Final End-to-End Pipeline Evaluation:")
for k, v in final_res.items():
    print(f"  - {k}: {v*100:.2f}%")

result_file = os.path.join(session_dirs["logs"], "result.txt")
if os.path.exists(result_file):
    print(f"\n✅ Full prediction file saved: {result_file}")

## 7. Display Step 2 Training Loss & Metrics Curve

In [ ]:
from IPython.display import Image, display
plot_path = os.path.join(session_dirs["plots"], "04_step2_training_loss_f1_curve.png")
if os.path.exists(plot_path):
    display(Image(plot_path))

print("✨ Step 2 finished! Proceed to '05_ACOS_Evaluation_and_Interactive_Inference.ipynb' for benchmark dashboard & interactive demo!")